# J-PCMCI+ Joint Effectome Sweep

This notebook mirrors the `jpcmciplus` simulation template, but is adapted to pooled effectome windows
from multiple recordings. For each `tau_max` and shared window index, it runs J-PCMCI+ across contexts
and saves a joint signed effectome matrix stack under `outputs/notebooks/jpcmciplus/`.

Before running, replace `WINDOWS_PATHS` with at least two compatible `windows.pkl` artifacts.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / 'src' / 'effectome').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate effectome project root')

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from effectome.experiments import (
    collapse_tigramite_results,
    find_project_root,
    normalize_multiple_recordings,
    plot_depth_summary,
    summarize_stack,
)
from effectome.utils.io import load_artifact, save_artifact, save_matrices
from tigramite import data_processing as pp
from tigramite.independence_tests.parcorr_mult import ParCorrMult
from tigramite.jpcmciplus import JPCMCIplus

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
WINDOWS_PATHS = [
    PROJECT_ROOT / 'outputs' / 'artifacts' / 'windows.pkl',
    # PROJECT_ROOT / 'outputs' / 'artifacts' / 'windows_fish2.pkl',
]
TAU_MIN = 1
TAU_MAX = 7
TAU_RANGE = list(range(TAU_MIN, TAU_MAX + 1))
PC_ALPHA = 0.05
EDGE_ALPHA = 0.05
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebooks' / 'jpcmciplus'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Window artifacts:')
for path in WINDOWS_PATHS:
    print(' ', path)
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
window_sets = [load_artifact(path) for path in WINDOWS_PATHS]
if len(window_sets) < 2:
    raise ValueError('J-PCMCI+ requires at least two window artifacts from separate recordings.')

n_windows = window_sets[0].n_windows
n_neurons = window_sets[0].n_neurons
for idx, windows in enumerate(window_sets):
    if windows.n_windows != n_windows:
        raise ValueError(f'Context {idx} has {windows.n_windows} windows; expected {n_windows}.')
    if windows.n_neurons != n_neurons:
        raise ValueError(f'Context {idx} has {windows.n_neurons} neurons; expected {n_neurons}.')

print(f'Loaded {len(window_sets)} contexts, {n_windows} shared windows, {n_neurons} neurons')


def estimate_jpcmci_stack(window_sets, tau_max: int) -> np.ndarray:
    matrices = []
    node_classification = {i: 'system' for i in range(n_neurons)}
    for window_idx in tqdm(range(n_windows), desc=f'J-PCMCI+ tau={tau_max}'):
        data_dict = normalize_multiple_recordings(
            [np.asarray(windows.segments[window_idx], dtype=np.float64) for windows in window_sets]
        )
        dataframe = pp.DataFrame(
            data=data_dict,
            analysis_mode='multiple',
            var_names=[str(i) for i in range(n_neurons)],
        )
        estimator = JPCMCIplus(
            dataframe=dataframe,
            cond_ind_test=ParCorrMult(significance='analytic'),
            node_classification=node_classification,
            verbosity=0,
        )
        results = estimator.run_jpcmciplus(
            tau_min=TAU_MIN,
            tau_max=tau_max,
            pc_alpha=PC_ALPHA,
        )
        matrices.append(collapse_tigramite_results(results, alpha=EDGE_ALPHA, tau_min=TAU_MIN))
    return np.asarray(matrices, dtype=np.float32)


In [ ]:
results = {}
summary_rows = []

for tau in TAU_RANGE:
    matrices = estimate_jpcmci_stack(window_sets, tau)
    save_matrices(matrices, OUTPUT_DIR / f'jpcmciplus_tau_{tau}.npz')
    results[tau] = matrices
    stats = summarize_stack(matrices)
    stats['tau'] = tau
    summary_rows.append(stats)

summary_df = pd.DataFrame(summary_rows).sort_values('tau').reset_index(drop=True)
summary_df.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
summary_df


In [ ]:
fig = plot_depth_summary(summary_df, 'tau', 'J-PCMCI+ Joint Effectome Summary vs tau_max')
fig.savefig(OUTPUT_DIR / 'jpcmciplus_tau_summary.png', dpi=150, bbox_inches='tight')
plt.show()

save_artifact(
    {'method': 'jpcmciplus', 'taus': TAU_RANGE, 'summary': summary_df.to_dict(orient='records')},
    OUTPUT_DIR / 'summary.pkl',
)
print('Saved notebook outputs to', OUTPUT_DIR)
